# Learn 2 Divide


In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from types import SimpleNamespace

from problems.problem_cvrp import CVRP, get_capacity, CVRPDataset
from agent.dnc import DNC, Divider
from utils.utils import move_to
from options import get_options
from hgs_solver import HGSSolver
from agent.ppo import PPO

import plotly.graph_objects as go

In [2]:
opts = get_options('')
opts.problem = 'cvrp' 
opts.wo_feature1 = False #feature1 corresponds to these two features : infeasibility_indicator_before_visit,infeasibility_indicator_after_visit. If wo_feature1 is False, then agent has those two features. 
opts.wo_feature3 = False #feature3 corresponds to exploration statistics (i think).
opts.wo_regular = False
opts.wo_bonus = False
opts.wo_RNN = False
opts.wo_MDP = True
opts.use_cuda = torch.cuda.is_available()
opts.stall_limit = 10
opts.val_m = 8
opts.graph_size = 400
opts.init_val_met = 'greedy'
opts.no_saving = True
opts.no_tb = True
opts.val_size = 128
opts.load_path = 'pre-trained/cvrp100.pt'
opts.device = torch.device("cuda" if opts.use_cuda else "cpu")
opts.no_progress_bar = False
opts.batch_size = 128
opts.T_max = 100
opts.record = False
opts.dummy_rate = 0.5
opts.dnc_n_splits = 4
opts.lr_divider = 1e-4
opts

print(f"Utilisation du device : {opts.device}")

Utilisation du device : cuda


In [3]:
from nets.divider_net import NeuralDivider, NeuralDividerImproved

global_problem = CVRP(
                        p_size = opts.graph_size,
                        init_val_met = opts.init_val_met,
                        with_assert = opts.use_assert,
                        DUMMY_RATE = opts.dummy_rate,
                        k = opts.k,
                        with_bonus = not opts.wo_bonus,
                        with_regular = not opts.wo_regular
                        )


divider = NeuralDivider(opts).to(opts.device)

agent = DNC(global_problem, opts, divider)
checkpoint_path = 'pre-trained/cvrp100.pt'
agent.load(checkpoint_path)

CVRP with 400 nodes and 200 dummy depots (total 600).
 Regulation: True Bonus: True Do assert: False.
 MAX 4-opt.

CVRP with 100 nodes and 50 dummy depots (total 150).
 Regulation: True Bonus: True Do assert: False.
 MAX 4-opt.

simpleMDP:  True
# params in Actor {'Total': 685140, 'Trainable': 685140}
 [*] Loading data from pre-trained/cvrp100.pt


In [4]:

train_dataset = CVRPDataset(size=opts.graph_size,
                      num_samples=6000,
                      filename = None,
                      DUMMY_RATE = opts.dummy_rate,
                      distribution='centered')

train_dataloader = torch.utils.data.DataLoader(train_dataset, batch_size=opts.batch_size, shuffle=False, pin_memory=True)

eval_dataset = CVRPDataset(size=opts.graph_size,
                      num_samples=800,
                      filename = None,
                      DUMMY_RATE = opts.dummy_rate,
                      distribution='centered')

eval_dataloader = torch.utils.data.DataLoader(eval_dataset, batch_size=128, shuffle=False, pin_memory=True)


from agent.divider_trainer import DividerTrainer
divider_trainer = DividerTrainer(divider,agent, opts)

6000 instances initialized.
800 instances initialized.


In [5]:
import time
start_time = time.time()
# divider_trainer.train(train_dataloader, eval_dataloader, n_epochs=51)
end_time = time.time()
print(f"Training took {end_time - start_time:.2f} seconds")

Training took 0.00 seconds


In [6]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import numpy as np

def plot_dnc(batch, results, rollout_output, n_splits, idx=0):
    """
    Visualisation interactive :
    - Gauche : Carte des routes (Vue locale, connexions inter-clients)
    - Droite : Courbe de convergence du coût total (Best So Far)
    """
    coords = batch['coordinates'][idx].cpu().numpy()
    full_route = results['routes'][idx].cpu().numpy()
    final_cost = results['total_cost'][idx].item()
    depot_pos = coords[0]
    
    # --- Récupération de l'historique du coût ---
    # rollout_output[1] : Tensor [Total_SubProblems, Iterations, 3]
    # On isole les sous-problèmes correspondants à notre instance 'idx'
    
    history_tensor = rollout_output[1] 
    start_split = idx * n_splits
    end_split = (idx + 1) * n_splits
    
    # On prend la composante 1 (Best Cost So Far)
    # Et on somme sur les n_splits pour avoir le vrai coût total de l'instance
    sub_costs = history_tensor[start_split:end_split, :, 1] # [n_splits, T+1]
    cost_evolution = sub_costs.sum(dim=0).cpu().numpy()     # [T+1]
    
    steps = np.arange(len(cost_evolution))

    # --- Initialisation Figure (1 ligne, 2 colonnes) ---
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=(f"Solution DNC (Coût Final: {final_cost:.2f})", "Convergence (Best So Far)"),
        column_widths=[0.6, 0.4],
        specs=[[{"type": "xy"}, {"type": "xy"}]]
    )

    # 1. Plot Solution (Map)
    
    # Clients
    fig.add_trace(go.Scatter(
        x=coords[1:, 0], y=coords[1:, 1],
        mode='markers',
        marker=dict(size=4, color='lightgray'),
        name='Clients',
        hoverinfo='skip',
        showlegend=False
    ), row=1, col=1)

    # Depots
    fig.add_trace(go.Scatter(
        x=[depot_pos[0]], y=[depot_pos[1]],
        mode='markers',
        marker=dict(symbol='square', size=12, color='red', line=dict(width=1, color='black')),
        name='Dépôt',
        hoverinfo='name',
        showlegend=False
    ), row=1, col=1)


    total_steps = len(full_route)
    steps_per_split = total_steps // n_splits
    
    colors = [
        '#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', 
        '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf'
    ]

    for i in range(n_splits):
        start_idx = i * steps_per_split
        end_idx = (i + 1) * steps_per_split
        segment_indices = full_route[start_idx:end_idx]
        
        x_vals = []
        y_vals = []
        hover_texts = []
        
        for node_idx in segment_indices:
            if node_idx == 0:
                x_vals.append(None)
                y_vals.append(None)
                hover_texts.append(None)
            else:
                c = coords[node_idx]
                x_vals.append(c[0])
                y_vals.append(c[1])
                hover_texts.append(str(node_idx))
        
        # Tracé route
        split_name = f"Split {i+1}"
        color = colors[i % len(colors)]
        
        fig.add_trace(go.Scatter(
            x=x_vals,
            y=y_vals,
            mode='lines+markers',
            line=dict(width=2, color=color),
            marker=dict(size=6, color=color),
            name=split_name,
            legendgroup=split_name,
            text=hover_texts,
            hovertemplate=f"<b>{split_name}</b><br>Client: %{{text}}<extra></extra>",
            connectgaps=False 
        ), row=1, col=1)

    # 2. Plot Convergence (Line)
    fig.add_trace(go.Scatter(
        x=steps, y=cost_evolution,
        mode='lines',
        line=dict(color='darkblue', width=2),
        name='Coût Total'
    ), row=1, col=2)

    # 3. Layout Adjustments
    fig.update_xaxes(range=[0, 1], showgrid=False, zeroline=False, scaleanchor="y", scaleratio=1, row=1, col=1)
    fig.update_yaxes(range=[0, 1], showgrid=False, zeroline=False, row=1, col=1)

    fig.update_xaxes(title_text="Itérations", row=1, col=2)
    fig.update_yaxes(title_text="Coût Total", row=1, col=2)
    
    fig.update_layout(
        width=1100, height=550,
        legend=dict(itemclick="toggleothers", itemdoubleclick="toggle"),
        template="plotly_white",
        plot_bgcolor='rgba(245,245,245,0.3)' # Fond très léger
    )

    fig.show()

In [9]:
divider.load_state_dict(torch.load(r'trained_divider\CVRP400\run_100ite_neuopt\best_model.pt', map_location=opts.device)['model_state_dict'])
agent = DNC(global_problem, opts, divider)
agent.load('pre-trained/cvrp100.pt')
visualisation_dataset = CVRPDataset(size=opts.graph_size,
                      num_samples=8,
                      filename = None,
                      DUMMY_RATE= opts.dummy_rate,
                      distribution='centered')
visualisation_dataloader = torch.utils.data.DataLoader(visualisation_dataset, batch_size=8, shuffle=False, pin_memory=True)
batch = next(iter(visualisation_dataloader))
batch = {k: v.to(opts.device, non_blocking=True) for k, v in batch.items()}
results, rollout_output = agent.solve(batch=batch, T=300, val_m=opts.val_m, stall_limit=opts.stall_limit, show_bar=not opts.no_progress_bar)
plot_dnc(batch, results, rollout_output, n_splits=opts.dnc_n_splits, idx=0)

CVRP with 100 nodes and 50 dummy depots (total 150).
 Regulation: True Bonus: True Do assert: False.
 MAX 4-opt.

simpleMDP:  True
# params in Actor {'Total': 685140, 'Trainable': 685140}
 [*] Loading data from pre-trained/cvrp100.pt
8 instances initialized.


DNC rollout: 100%|████████████████████| 300/300 [00:53<00:00,  5.62it/s]


In [ ]:
plot_dnc(batch, results, rollout_output, n_splits=opts.dnc_n_splits, idx=5)

In [ ]:
#save internal state of divider
divider_state = divider.state_dict()
torch.save(divider_state, 'divider_state.pt')

In [10]:
print(f"Lancement de l'inférence (Split en {opts.dnc_n_splits} sous-problèmes)...")
    
# On appelle la méthode solve que nous avons ajoutée à la classe DNC
# T=100 ou 200 itérations de PPO pour raffiner la solution
agent_ppo = PPO(global_problem, opts)
agent_ppo.eval()
agent_ppo.load(opts.load_path)
hgss = HGSSolver(global_problem, time_limit=20)

beam_divider = Divider(global_problem, opts)
agent_dnc_only = DNC(global_problem, opts, beam_divider)
agent_dnc_only.load(opts.load_path)
ppo_total_cost = 0.0
dnc_total_cost = 0.0
dnc_nn_total_cost = 0.0
hgss_total_cost = 0.0

    
out = agent_ppo.rollout(batch=batch,problem=global_problem, T=opts.T_max, val_m=opts.val_m, stall_limit=opts.stall_limit, show_bar=not opts.no_progress_bar, record=True)   # PPO
ppo_res = out[0]
results, rollout_output = agent_dnc_only.solve(batch=batch, T=opts.T_max, val_m=opts.val_m, stall_limit=opts.stall_limit, show_bar=not opts.no_progress_bar)                # DNC avec Divider classique
results_nn, rollout_output_nn = agent.solve(batch=batch, T=opts.T_max, val_m=opts.val_m, stall_limit=opts.stall_limit, show_bar=not opts.no_progress_bar)                   # DNC avec Divider neuronal
hgss_results = np.array(hgss(batch))                                                                                                                                        # HGSS

ppo_total_cost += ppo_res.sum().item()
dnc_total_cost += results['total_cost'].sum().item()
dnc_nn_total_cost += results_nn['total_cost'].sum().item()
hgss_total_cost += np.sum(hgss_results)

dnc_mean_cost = dnc_total_cost / 8
dnc_nn_mean_cost = dnc_nn_total_cost / 8
hgss_mean_cost = hgss_total_cost / 8
ppo_mean_cost = ppo_total_cost / 8
    

# ==========================================
# 5. RÉSULTATS & VISUALISATION
# ==========================================
print(f" Mean cost DNC : {dnc_mean_cost:.4f}")
print(f" Mean cost DNC with NN divider : {dnc_nn_mean_cost:.4f}")
print(f" Mean cost PPO : {ppo_mean_cost:.4f}")
print(f" Mean cost HGSS : {hgss_mean_cost:.4f}")
print(f'Gap : {((dnc_mean_cost - hgss_mean_cost) / hgss_mean_cost) * 100:.2f}%')
# Visualisation
plot_dnc(batch, results, rollout_output, n_splits=opts.dnc_n_splits, idx=0)
plot_dnc(batch, results_nn, rollout_output_nn, n_splits=opts.dnc_n_splits, idx=0)

Lancement de l'inférence (Split en 4 sous-problèmes)...
simpleMDP:  True
# params in Actor {'Total': 685140, 'Trainable': 685140}
# params in Critic {'Total': 191107, 'Trainable': 191107}
Distributed: False
 [*] Loading data from pre-trained/cvrp100.pt
 [*] Model loaded successfully (RNG states ignored).
CVRP with 100 nodes and 50 dummy depots (total 150).
 Regulation: True Bonus: True Do assert: False.
 MAX 4-opt.

simpleMDP:  True
# params in Actor {'Total': 685140, 'Trainable': 685140}
 [*] Loading data from pre-trained/cvrp100.pt


DNC rollout: 100%|████████████████████| 100/100 [00:17<00:00,  5.68it/s]


 Mean cost DNC : 25.4531
 Mean cost DNC with NN divider : 37.0925
 Mean cost PPO : 32.9557
 Mean cost HGSS : 23.4428
Gap : 8.58%
